# DINOv3 SAT sur HEALPix — réglage sur une seule date

Ce notebook fait tourner `GetDINOV3SAT` sur **une date** du store Sentinel-2
HEALPix et donne de quoi comprendre ce que fait le réseau avant de faire
confiance à une classification.

Le k-means tourne **sur les coordonnées UMAP**, pas sur les 1024 dimensions de
DINOv3. Raison : k-means cherche des groupes sphériques de variance comparable,
ce que les embeddings bruts ne forment pas — ils vivent sur une variété courbe où
les distances euclidiennes ne veulent pas dire grand-chose. UMAP déplie cette
variété et sépare les paquets, ce que k-means sait alors découper.

Le prix à en connaître : UMAP déforme les distances globales, la taille et
l'écartement des groupes sur le tracé ne sont pas interprétables, et le résultat
dépend de `n_neighbors`, `min_dist` et de la graine aléatoire. Deux réglages
différents donnent deux partitions différentes, toutes deux « valables ». D'où
§11, qui compare la partition UMAP à celle obtenue directement sur DINOv3 et à
un k-means sur la couleur seule.

L'ordre des cellules évite de recalculer ce qui coûte cher :

| étape | coût | cellule |
|---|---|---|
| lecture de la date | ~1 min la 1re fois, puis cache disque | §2 |
| découpage en tuiles | secondes | §3 |
| passage dans DINOv3 | 10 s – 2 min | §4 |
| PCA de contrôle | instantané | §5 |
| **UMAP** | 10 s – 1 min | **§6 — à rejouer si on change `n_neighbors`** |
| **k-means** | instantané | **§7 — à rejouer pour régler `K`** |
| cartes, inspection | secondes | §8 – §11 |

## 1. Paramètres

In [ ]:
import os, sys, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))          # pour importer le script voisin
sys.path.insert(0, str(Path.cwd().parent))   # pour importer healpix_analyse en dev

# --------------------------------------------------------------------------
# À RÉGLER
# --------------------------------------------------------------------------
ZARR    = "https://data-taos.ifremer.fr/EGU25_CFOSAT/Sentinel2_test.zarr"
CACHE   = os.path.expanduser("~/s2_cache")   # None pour ne rien mettre en cache
WEIGHTS = os.path.expanduser("~/QTRACE/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth")

TIME_INDEX   = 0      # la date étudiée (liste imprimée en §2)
TILE_LEVELS  = 8      # côté des tuiles DINO = 2**TILE_LEVELS px (8 -> 256 px)
MIN_COVERAGE = 1.0    # 1.0 = uniquement les tuiles parent COMPLÈTES (aucun NaN)

# UMAP (§6)
UMAP_DIM       = 2    # dimensions où le k-means travaille. 2 = ce qu'on voit ;
                      # 5 à 10 sépare souvent mieux, les 2 premières servant à l'affichage
UMAP_NEIGHBORS = 30   # petit (10) = détail local ; grand (100) = structure globale
UMAP_MIN_DIST  = 0.0  # 0.0 tasse les groupes, ce qui aide le k-means

# k-means (§7)
K = 12

FAKE = False          # True = réseau aléatoire, pour tester la mécanique sans poids
# --------------------------------------------------------------------------

from healpix_analyse.dino import GetDINOV3SAT, load_dinov3_sat, nested_to_tiles
from dino_umap_sentinel2 import (
    FakeDino, RGB, categorical_cmap, cell_dim, healpix_level, open_store, rgb_at,
)

print("bandes RGB utilisées :", RGB)

## 2. Lecture d'une date

La première exécution télécharge le chunk de la date (quelques centaines de Mo) ;
avec `CACHE`, les suivantes le relisent sur disque.

In [ ]:
ds    = open_store(ZARR)
level = healpix_level(ds)
cell_id = ds["cell_ids"].values.astype(np.int64)
dates = [str(d)[:10] for d in ds["time"].values]

print(f"niveau HEALPix {level}, {cell_id.size} cellules, {len(dates)} dates")
print("dates :", ", ".join(f"{i}:{d}" for i, d in enumerate(dates[:12])), "...")

t0 = time.time()
rgb = rgb_at(ds, TIME_INDEX, cache=CACHE)          # [N, 3] réflectances dans [0, 1]
print(f"date {TIME_INDEX} = {dates[TIME_INDEX]} lue en {time.time()-t0:.0f}s ; "
      f"NaN : {100*np.isnan(rgb).any(1).mean():.1f}% des cellules")

## 3. Découpage en tuiles — *ce que le réseau va vraiment voir*

C'est l'étape à regarder en premier quand un résultat semble incompréhensible.
`nested_to_tiles` replie chaque cellule de `parent_level` en une image carrée
exacte (aucun rééchantillonnage). Avec `fill="nan"` on voit les trous tels quels.

Rappel d'orientation : le nord d'une face HEALPix pointe vers le **coin
haut-droit** de l'image, pas vers le haut. Une image qui « penche » de 45° est
normale.

In [ ]:
parent_level = level - TILE_LEVELS
tiles, parent_ids, valid, coverage = nested_to_tiles(
    rgb, cell_id, level, parent_level, fill="nan")

keep = coverage >= MIN_COVERAGE
print(f"{tiles.shape[0]} tuiles de {tiles.shape[-1]} px au niveau {parent_level} ; "
      f"{keep.sum()} complètes à >= {MIN_COVERAGE}")
print("couverture : min %.3f  médiane %.3f  max %.3f"
      % (coverage.min(), np.median(coverage), coverage.max()))

fig, ax = plt.subplots(figsize=(6, 2.4))
ax.hist(coverage, bins=40, color="0.4")
ax.axvline(MIN_COVERAGE, color="crimson", lw=2, label=f"MIN_COVERAGE = {MIN_COVERAGE}")
ax.set_xlabel("fraction de pixels utilisables par tuile"); ax.set_ylabel("tuiles")
ax.legend(); plt.show()

In [ ]:
def show_tiles(imgs, titles=None, ncol=6, size=2.1, cmap=None, **kw):
    """Planche de vignettes."""
    n = len(imgs)
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(size * ncol, size * nrow), squeeze=False)
    for i, a in enumerate(axes.ravel()):
        a.set_axis_off()
        if i >= n:
            continue
        a.imshow(imgs[i], cmap=cmap, **kw)
        if titles is not None:
            a.set_title(titles[i], fontsize=8)
    fig.tight_layout()
    return fig

def to_rgb(tile, p_low=2, p_high=98):
    """[3, S, S] -> image affichable, étirée sur des percentiles (NaN en noir)."""
    img = np.transpose(tile, (1, 2, 0))
    lo, hi = np.nanpercentile(img, [p_low, p_high])
    return np.clip((img - lo) / max(hi - lo, 1e-6), 0, 1)

idx = np.where(keep)[0][:12]
show_tiles([to_rgb(tiles[i]) for i in idx],
           [f"{parent_ids[i]}  cov={coverage[i]:.3f}" for i in idx])
plt.show()

Si ces vignettes ne ressemblent pas à des paysages, inutile d'aller plus loin :
le problème est en amont (bandes, échelle des réflectances, niveau HEALPix).
Augmenter `TILE_LEVELS` si les tuiles sont trop petites pour montrer une
structure.

## 4. Passage dans DINOv3

`return_patches=True` donne un vecteur par cellule de `level - 4`
(un patch de 16 px), c'est ce qui sert à la classification.

In [ ]:
model = FakeDino() if FAKE else load_dinov3_sat("dinov3_vitl16", WEIGHTS)

t0 = time.time()
res = GetDINOV3SAT(
    rgb, cell_id, level, parent_level,
    model=model, return_patches=True, min_coverage=MIN_COVERAGE,
)
G = int(round(np.sqrt(res.patch_embedding.shape[0] / max(res.cell_id.size, 1))))
E = res.patch_embedding.reshape(res.cell_id.size, G, G, -1)     # [tuile, ligne, colonne, D]

print(f"{res.cell_id.size} tuiles -> {res.patch_embedding.shape} en {time.time()-t0:.0f}s")
print(f"grille de patches : {G}x{G} par tuile, dimension {E.shape[-1]}, "
      f"niveau des patches {res.patch_level}")

# les tuiles gardées, dans le même ordre que res.cell_id
sel = np.searchsorted(parent_ids, res.cell_id)
kept_tiles = tiles[sel]

# vecteurs normalisés L2 : c'est l'entrée de tout ce qui suit
z = res.patch_embedding.astype(np.float32)
z = z / (np.linalg.norm(z, axis=1, keepdims=True) + 1e-8)

# couleur moyenne (réflectance) de chaque patch, utile pour interpréter les groupes
S = kept_tiles.shape[-1]
px = kept_tiles.reshape(res.cell_id.size, 3, G, S // G, G, S // G).mean(axis=(3, 5))
px = np.transpose(px, (0, 2, 3, 1)).reshape(-1, 3)               # [patch, 3]

## 5. Les features ont-elles un sens ? — PCA des tokens

Le test standard sur DINO : projeter les tokens de patch sur leurs **3 premières
composantes principales** et les afficher en RGB. Si le réseau « voit » quelque
chose, cette image fait apparaître les objets (champs, forêt, routes, eau) sans
aucune supervision. Si elle ressemble à du bruit, le problème vient des
embeddings et ni UMAP ni k-means n'y changeront rien.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=3, random_state=0).fit(z)
proj = pca.transform(z).reshape(res.cell_id.size, G, G, 3)

lo, hi = np.percentile(proj, [2, 98])
pca_rgb = np.clip((proj - lo) / (hi - lo), 0, 1)
print("variance expliquée par les 3 premières composantes :",
      np.round(pca.explained_variance_ratio_, 3))

n = min(6, res.cell_id.size)
fig, axes = plt.subplots(2, n, figsize=(2.4 * n, 5.2), squeeze=False)
for j in range(n):
    axes[0, j].imshow(to_rgb(kept_tiles[j])); axes[0, j].set_axis_off()
    axes[0, j].set_title(f"RGB  {res.cell_id[j]}", fontsize=8)
    axes[1, j].imshow(pca_rgb[j], interpolation="nearest"); axes[1, j].set_axis_off()
    axes[1, j].set_title("PCA des tokens", fontsize=8)
fig.tight_layout(); plt.show()

## 6. UMAP — **à rejouer si on change `UMAP_NEIGHBORS` / `UMAP_MIN_DIST`**

C'est ici que la variété est dépliée. `n_neighbors` est le paramètre qui change
le plus le résultat : petit, UMAP conserve le voisinage immédiat et fabrique
beaucoup de petits paquets ; grand, il privilégie la structure d'ensemble et en
fait quelques gros.

In [ ]:
import umap

t0 = time.time()
reducer = umap.UMAP(n_components=max(2, UMAP_DIM), n_neighbors=UMAP_NEIGHBORS,
                    min_dist=UMAP_MIN_DIST, metric="cosine", random_state=0)
u = reducer.fit_transform(z)
print(f"UMAP {u.shape} en {time.time()-t0:.0f}s "
      f"(n_neighbors={UMAP_NEIGHBORS}, min_dist={UMAP_MIN_DIST})")

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.scatter(u[:, 0], u[:, 1],
           c=np.clip(px / max(np.nanpercentile(px, 98), 1e-6), 0, 1), s=3, alpha=0.8)
ax.set_title("UMAP, couleur = couleur RGB réelle du patch")
ax.set_xticks([]); ax.set_yticks([]); plt.show()

Ce tracé, coloré par la couleur réelle, se lit avant tout k-means : si les
paquets séparés par UMAP correspondent à des couleurs franchement différentes,
DINOv3 n'encode guère plus que la radiométrie ; s'ils mélangent des couleurs
proches, c'est qu'il sépare des **textures**, et c'est là que la méthode
apporte quelque chose.

## 7. k-means sur l'UMAP — **la cellule à rejouer pour régler `K`**

Rien de coûteux ici. On peut relancer autant de fois qu'on veut en changeant `K`.

`ALGO = "hdbscan"` est une alternative qui **trouve le nombre de groupes toute
seule** et laisse en dehors les points ambigus (étiquette −1) : utile quand on
ne sait pas quoi mettre dans `K`.

In [ ]:
from sklearn.cluster import KMeans

K    = 12            # <-- à modifier et réexécuter
ALGO = "kmeans"      # "kmeans" ou "hdbscan"

if ALGO == "kmeans":
    model_c = KMeans(n_clusters=K, n_init=10, random_state=0).fit(u)
    labels = model_c.labels_
    centers = model_c.cluster_centers_
else:
    from sklearn.cluster import HDBSCAN
    model_c = HDBSCAN(min_cluster_size=max(25, u.shape[0] // 200)).fit(u)
    labels = model_c.labels_
    K = labels.max() + 1
    centers = np.stack([u[labels == k].mean(0) for k in range(K)]) if K else np.zeros((0, u.shape[1]))
    print(f"HDBSCAN a trouvé {K} groupes, {100*(labels==-1).mean():.1f}% de points non classés")

lab = np.where(labels < 0, K, labels).reshape(res.cell_id.size, G, G)   # -1 -> classe "hors groupe"
cmap = categorical_cmap(max(K, 1))

print(f"\n k | taille  |   R     V     B   (réflectance moyenne du groupe)")
for k in range(K):
    m = labels == k
    r, g, b = np.nanmean(px[m], axis=0)
    print(f"{k:2d} | {m.sum():6d}  | {r:.3f} {g:.3f} {b:.3f}")

try:
    from sklearn.metrics import silhouette_score
    sub = np.random.default_rng(0).choice(u.shape[0], size=min(5000, u.shape[0]), replace=False)
    ok = labels[sub] >= 0
    print(f"\nsilhouette dans l'espace UMAP (échantillon) : "
          f"{silhouette_score(u[sub][ok], labels[sub][ok]):.3f}")
except Exception as e:
    print("silhouette non calculée :", e)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
axes[0].scatter(u[:, 0], u[:, 1], c=np.where(labels < 0, K, labels), cmap=cmap,
                s=3, alpha=0.7, vmin=-0.5, vmax=max(K, 1) - 0.5)
axes[0].set_title(f"UMAP, couleur = groupe ({ALGO}, K={K})")
axes[1].scatter(u[:, 0], u[:, 1],
                c=np.clip(px / max(np.nanpercentile(px, 98), 1e-6), 0, 1), s=3, alpha=0.8)
axes[1].set_title("UMAP, couleur = couleur RGB réelle")
for a in axes:
    a.set_xticks([]); a.set_yticks([])
plt.show()

## 8. Les groupes replacés sur les tuiles

In [ ]:
n = min(6, res.cell_id.size)
fig, axes = plt.subplots(3, n, figsize=(2.4 * n, 7.6), squeeze=False)
for j in range(n):
    axes[0, j].imshow(to_rgb(kept_tiles[j])); axes[0, j].set_title(f"RGB {res.cell_id[j]}", fontsize=8)
    axes[1, j].imshow(pca_rgb[j], interpolation="nearest"); axes[1, j].set_title("PCA", fontsize=8)
    axes[2, j].imshow(lab[j], cmap=cmap, vmin=-0.5, vmax=max(K, 1) - 0.5, interpolation="nearest")
    axes[2, j].set_title(f"{ALGO} K={K}", fontsize=8)
    for a in axes[:, j]:
        a.set_axis_off()
fig.tight_layout(); plt.show()

Comment lire ces trois lignes :

- si **PCA** montre les structures mais que les groupes les hachent, `K` est trop
  grand, ou `UMAP_NEIGHBORS` trop petit (UMAP a fragmenté la variété) ;
- si des objets visiblement différents tombent dans le même groupe, `K` est trop
  petit, ou `UMAP_NEIGHBORS` trop grand ;
- si les groupes suivent un dégradé de luminosité plutôt que les objets, les
  embeddings sont dominés par l'éclairement : voir §11.

### 8b. À quoi ressemble un groupe donné ?

Extrait les patches de 16 px les plus proches du centre du groupe, dans l'espace
UMAP.

In [ ]:
CLUSTER = 0     # <-- groupe à inspecter

d = np.linalg.norm(u - centers[CLUSTER], axis=1)
d[labels != CLUSTER] = np.inf
best = np.argsort(d)[:24]

p = S // G                                  # côté d'un patch en pixels
ti, ri, ci = np.unravel_index(best, (res.cell_id.size, G, G))
patches = [to_rgb(kept_tiles[t][:, r * p:(r + 1) * p, c * p:(c + 1) * p], 5, 95)
           for t, r, c in zip(ti, ri, ci)]
show_tiles(patches, [f"tuile {res.cell_id[t]}" for t in ti], ncol=8, size=1.5)
plt.suptitle(f"groupe {CLUSTER} — {(labels == CLUSTER).sum()} patches", y=1.02)
plt.show()

## 9. Carte HEALPix des groupes

Avec `healpix_plot`, en lon/lat. Les patches sont au niveau `res.patch_level`.

In [ ]:
import cartopy.crs as ccrs
import healpix_plot

grid_px    = healpix_plot.HealpixGrid(level=level, indexing_scheme="nested", ellipsoid="WGS84")
grid_patch = healpix_plot.HealpixGrid(level=res.patch_level, indexing_scheme="nested",
                                      ellipsoid="WGS84")

fig, axes = plt.subplots(1, 2, figsize=(15, 6),
                         subplot_kw={"projection": ccrs.PlateCarree()}, layout="constrained")
hi = np.nanpercentile(rgb, 98)
healpix_plot.plot(cell_id, rgb, healpix_grid=grid_px, sampling_grid={"shape": 900},
                  ax=axes[0], rgb_clip=(0.0, float(hi)), axis_labels="none",
                  title=f"{dates[TIME_INDEX]}  RGB (niveau {level})")
mp = healpix_plot.plot(res.patch_cell_id, np.where(labels < 0, np.nan, labels).astype(np.float32),
                       healpix_grid=grid_patch, sampling_grid={"shape": 900},
                       ax=axes[1], cmap=cmap, vmin=-0.5, vmax=max(K, 1) - 0.5,
                       axis_labels="none",
                       title=f"{ALGO} K={K} sur UMAP (niveau {res.patch_level})")
fig.colorbar(mp, ax=axes[1], shrink=0.7, ticks=range(K))
plt.show()

## 10. Cohérence spatiale des groupes

Un indicateur simple et parlant : la fraction de patches voisins (dans la même
tuile) qui portent la même étiquette. Une classification qui a du sens
géographique est nettement au-dessus du hasard (`1/K`).

In [ ]:
lab_full = labels.reshape(res.cell_id.size, G, G)
same_h = (lab_full[:, :, 1:] == lab_full[:, :, :-1]).mean()
same_v = (lab_full[:, 1:, :] == lab_full[:, :-1, :]).mean()
print(f"voisins de même étiquette : {0.5*(same_h+same_v):.3f}   (hasard : {1/max(K,1):.3f})")

## 11. Trois espaces de classification comparés

La même méthode k-means appliquée à trois représentations : les coordonnées
UMAP, les embeddings DINOv3 bruts, et la couleur moyenne seule. L'indice de Rand
ajusté chiffre à quel point deux partitions se recoupent (1 = identiques,
0 = sans rapport).

Lecture utile : si **UMAP** et **couleur seule** donnent la même chose,
l'embedding n'apporte pas de texture. Si **UMAP** et **DINO brut** diffèrent
beaucoup, c'est UMAP qui fabrique la structure — ce n'est pas rédhibitoire, mais
les groupes dépendent alors de ses réglages, à vérifier en changeant la graine.

In [ ]:
from sklearn.metrics import adjusted_rand_score

lab_dino = KMeans(n_clusters=K, n_init=10, random_state=0).fit_predict(z)
lab_rgb  = KMeans(n_clusters=K, n_init=10, random_state=0).fit_predict(np.nan_to_num(px))

print(f"UMAP  vs  DINO brut       : {adjusted_rand_score(labels, lab_dino):.3f}")
print(f"UMAP  vs  couleur seule   : {adjusted_rand_score(labels, lab_rgb):.3f}")
print(f"DINO  vs  couleur seule   : {adjusted_rand_score(lab_dino, lab_rgb):.3f}")

# stabilité de la partition UMAP à une autre graine
u2 = umap.UMAP(n_components=max(2, UMAP_DIM), n_neighbors=UMAP_NEIGHBORS,
               min_dist=UMAP_MIN_DIST, metric="cosine", random_state=1).fit_transform(z)
lab_u2 = KMeans(n_clusters=K, n_init=10, random_state=0).fit_predict(u2)
print(f"UMAP graine 0  vs  graine 1 : {adjusted_rand_score(labels, lab_u2):.3f}  "
      "(proche de 1 = partition stable)")

n = min(5, res.cell_id.size)
fig, axes = plt.subplots(4, n, figsize=(2.4 * n, 10), squeeze=False)
rows = [("RGB", None), ("UMAP", lab_full),
        ("DINO brut", lab_dino.reshape(res.cell_id.size, G, G)),
        ("couleur seule", lab_rgb.reshape(res.cell_id.size, G, G))]
for j in range(n):
    for i, (name, arr) in enumerate(rows):
        if arr is None:
            axes[i, j].imshow(to_rgb(kept_tiles[j]))
        else:
            axes[i, j].imshow(arr[j], cmap=cmap, vmin=-0.5, vmax=max(K, 1) - 0.5,
                              interpolation="nearest")
        axes[i, j].set_title(name, fontsize=8); axes[i, j].set_axis_off()
fig.tight_layout(); plt.show()

## 12. Aide-mémoire de réglage

| symptôme | paramètre | où reprendre |
|---|---|---|
| trop peu de groupes, tout est mélangé | `K` | §7 seulement |
| on ne sait pas quoi mettre dans `K` | `ALGO = "hdbscan"` | §7 seulement |
| groupes hachés, sans cohérence spatiale (§10 proche de 1/K) | `UMAP_NEIGHBORS` plus grand | §6 |
| un seul gros paquet dans l'UMAP | `UMAP_NEIGHBORS` plus petit, `UMAP_MIN_DIST = 0.0` | §6 |
| partition instable d'une graine à l'autre (§11) | `UMAP_DIM` 5–10, ou classer sur DINO brut | §6 |
| tuiles trop petites, pas de contexte | `TILE_LEVELS` (8 → 256 px, 10 → 1024 px) | §3 |
| presque aucune tuile gardée | `MIN_COVERAGE` (0.95 tolère quelques nuages) | §3 |
| carte bruitée alors que la PCA (§5) est déjà du bruit | rien à régler ici | le problème est les embeddings |
| résultat identique à la couleur (§11) | `TILE_LEVELS` plus grand, vérifier la normalisation SAT | §3 |

Deux constantes valent la peine d'être vérifiées contre le *model card* de la
version des poids téléchargée : `SAT493M_MEAN` et `SAT493M_STD` dans
`healpix_analyse/dino.py`. Une normalisation fausse dégrade les features sans
produire d'erreur visible.

Une fois les réglages trouvés, on passe aux 88 dates :

```bash
python dino_umap_sentinel2.py --weights ... --cache ~/s2_cache \
    --clusters K --tile-levels 8 --umap-neighbors 30 --umap-min-dist 0.0
```

(le script classe lui aussi sur l'UMAP par défaut ; `--cluster-space dino` pour
revenir aux embeddings bruts).